# Part 1: Deploy Nemotron on DGX Spark

**Nemotron** is NVIDIA's family of open language models, built for reasoning, tool use and long-context chat. In this lab you will run a Nemotron model on your own DGX Spark.


## 1. Prerequisites

- Dell Pro Max GB10 (or DGX Spark) with at least 100 GB of free disk
- Docker with GPU access (NVIDIA Container Toolkit)
- Access to Hugging Face and container registries

Run the next cell to check your system. It confirms the GPU, disk space, Docker with GPU access, and network access to Hugging Face and Docker Hub. Every line should say `[OK]` before you continue. (`HTTP 401` next to Docker Hub is normal.)


In [ ]:
# Prerequisite check: every line should print [OK]
import platform, shutil, subprocess
import requests

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def check(label, ok, detail=""):
    print(f"[{'OK' if ok else 'FAIL'}] {label}" + (f"  {detail}" if detail else ""))

# Hardware
gpu = run("nvidia-smi -L").stdout.strip()
check("GPU", "GB10" in gpu, gpu)
check("Architecture", platform.machine() == "aarch64", platform.machine())

# Disk
free_gb = shutil.disk_usage("/").free / 1e9
check("Free disk (want 100+ GB)", free_gb >= 100, f"{free_gb:.0f} GB")

# Docker with GPU access
docker = run("docker info --format '{{json .Runtimes}}'")
check("Docker usable", docker.returncode == 0, docker.stderr.strip())
gpu_in_docker = run("docker run --rm --gpus all ubuntu:24.04 nvidia-smi -L")
check("GPU visible inside Docker", gpu_in_docker.returncode == 0 and "GB10" in gpu_in_docker.stdout,
      gpu_in_docker.stderr.strip()[-200:] if gpu_in_docker.returncode else "")

# Network access
for name, url in [("Hugging Face", "https://huggingface.co"),
                  ("Docker Hub", "https://registry-1.docker.io/v2/")]:
    try:
        r = requests.get(url, timeout=5)
        check(name, r.status_code < 500, f"HTTP {r.status_code}")
    except Exception as e:
        check(name, False, type(e).__name__)


## 2. Choose a Nemotron deployment

The GB10 has 128 GB of unified memory shared between CPU and GPU. The model you choose, and how much memory you let the server reserve, decide how much is left for everything else.

| | Nemotron Nano | Nemotron Super |
|---|---|---|
| Size | 30B parameters (3B active) | 120B parameters (12B active) |
| Precision | FP8 | NVFP4 |
| Serving engine | vLLM in Docker | vLLM or TensorRT-LLM in Docker |
| Memory | About 33 GB of weights. The server reserves more for working memory (`--gpu-memory-utilization`, see section 10) | Runs on one Spark, but the reference setup uses 90% of the unified memory |

This lab uses **Nano**.

Want to run Super on a Spark? See the [Nemotron 3 Super Spark deployment guide](https://github.com/NVIDIA-NeMo/nemotron/tree/main/usage-cookbook/Nemotron-3-Super/SparkDeploymentGuide).


## 3. Download the model

The weights for **NVIDIA-Nemotron-3-Nano-30B-A3B-FP8** are hosted on Hugging Face. The next cell downloads them to `~/models/nemotron-3-nano-fp8` on your system. This is the longest step in the lab, so it is fine to leave it running. Later runs skip files that are already downloaded.


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8"
MODEL_DIR = Path.home() / "models" / "nemotron-3-nano-fp8"

snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR)

size_gb = sum(f.stat().st_size for f in MODEL_DIR.rglob("*") if f.is_file()) / 1e9
print(f"Downloaded to {MODEL_DIR}  ({size_gb:.1f} GB)")


## 4. Start the model server

The model is served by **vLLM**, which runs in a Docker container and listens on port 8000. The next cell starts it in the background, so the notebook stays free to use.

| Flag | What it does |
|---|---|
| `--served-model-name nemotron` | The model name you use in requests |
| `--max-model-len 32768` | Longest conversation the server accepts, in tokens |
| `--max-num-seqs 8` | How many requests it handles at once |
| `--gpu-memory-utilization 0.8` | Share of the unified memory vLLM may use |
| `--reasoning-parser` | Separates the model's thinking from its answer |
| `--tool-call-parser` | Turns the model's tool requests into structured calls |

The cell returns right away, but the model takes time to load. The next section waits until the server is ready.

If the server runs out of memory, lower `--gpu-memory-utilization` to `0.70`, then lower `--max-model-len`, and run the cell again.


In [ ]:
%%bash
docker rm -f vllm-nemotron-nano 2>/dev/null
docker run -d \
  --gpus all --ipc=host --shm-size=16g -p 8000:8000 \
  --name vllm-nemotron-nano \
  -v ~/models/nemotron-3-nano-fp8:/model:ro \
  vllm/vllm-openai:v0.20.0 /model \
  --served-model-name nemotron \
  --max-num-seqs 8 \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.8 \
  --trust-remote-code \
  --enable-prefix-caching \
  --reasoning-parser nemotron_v3 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder


## 5. Wait for the server

Loading the model takes about 4 to 5 minutes. The next cell shows a live timer and tells you when the server is ready.


In [ ]:
import time, requests

start = time.time()
for _ in range(120):  # up to 10 minutes
    try:
        models = [m["id"] for m in requests.get("http://localhost:8000/v1/models", timeout=3).json()["data"]]
        print(f"\nServer ready after {time.time() - start:.0f}s. Model: {models}")
        break
    except Exception:
        print(f"\rLoading the model... {time.time() - start:.0f}s", end="", flush=True)
        time.sleep(5)
else:
    print("\nNot ready after 10 minutes. Check: docker logs --tail 40 vllm-nemotron-nano")


## 6. Your first request

The server speaks the same API format as OpenAI, so we use the `openai` Python library and point it at your system. The model's reasoning comes back separately from its final answer.


In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

resp = client.chat.completions.create(
    model="nemotron",
    messages=[{"role": "user", "content": "What is 17 * 24? Answer briefly."}],
    max_tokens=500,
)

msg = resp.choices[0].message
reasoning = getattr(msg, "reasoning_content", None) or getattr(msg, "reasoning", None)
print("Reasoning:", (reasoning or "")[:300] or "(none)")
print("\nAnswer:", msg.content)
print("\nTokens:", resp.usage.completion_tokens)


## 7. How fast is it?

The next cell sends one request and measures how many tokens per second your system generates.

**What to look for:** the number of tokens per second your own system generates on one request.


In [ ]:
import time

start = time.time()
resp = client.chat.completions.create(
    model="nemotron",
    messages=[{"role": "user", "content": "Write a short story about a robot learning to paint."}],
    max_tokens=400,
)
elapsed = time.time() - start

tokens = resp.usage.completion_tokens
print(f"{tokens} tokens in {elapsed:.1f}s  =  {tokens / elapsed:.1f} tokens/second")


## 8. Streaming

Instead of waiting for the whole reply, we can print it as the model produces it. Everything here runs on your own hardware.

**What to look for:** the reply prints as the model produces it, served from your own system at `localhost:8000`.


In [ ]:
SHOW_THINKING = False   # set to True to watch the model think first

stream = client.chat.completions.create(
    model="nemotron",
    messages=[{"role": "user", "content": "Write a 3-sentence email to a customer explaining why running AI on their own hardware keeps their data private. Do not include a subject line."}],
    max_tokens=1000,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta
    thinking = getattr(delta, "reasoning_content", None) or getattr(delta, "reasoning", None)
    if thinking and SHOW_THINKING:
        print(thinking, end="", flush=True)
    if delta.content:
        print(delta.content, end="", flush=True)
print()


## 9. Long context

Most AI demos use a short question. Real customer documents are long: contracts, manuals, policies. The next cell gives the model an entire file of about 100,000 characters (about 21,000 tokens) and asks for a summary. The first run took about 20 seconds in testing, because the model reads the whole file before it answers. If you run the cell again, it is quicker, because the server has already seen the file.

**What to look for:** the whole file, about 21,000 tokens, goes in one request. The server accepts up to 32,768 tokens per request (`--max-model-len 32768`).



In [ ]:
import argparse, time
from pathlib import Path

doc = Path(argparse.__file__).read_text()

start = time.time()
resp = client.chat.completions.create(
    model="nemotron",
    messages=[{"role": "user",
               "content": f"{doc}\n\nSummarize what the code above does in 3 sentences for a non-technical reader."}],
    max_tokens=3000,
)
elapsed = time.time() - start

choice = resp.choices[0]
print((choice.message.content or "(no answer)").strip())
print(f"\nRead {resp.usage.prompt_tokens:,} tokens in {elapsed:.0f}s, finish reason: {choice.finish_reason}")


## 10. Memory

The Dell Pro Max GB10 has 128 GB of unified memory (about 125 GB usable), shared between the CPU and the GPU. A 30B-parameter model and its working memory fit in it. The server reserved about 80% of the memory when it started (`--gpu-memory-utilization 0.8`), which is why most of it shows as in use below.


In [ ]:
def meminfo_gb():
    info = {}
    for line in open("/proc/meminfo"):
        key, value = line.split(":")
        info[key] = int(value.split()[0]) / 1e6   # kB -> GB
    return info

m = meminfo_gb()
total, available = m["MemTotal"], m["MemAvailable"]
print(f"Total memory:     {total:6.1f} GB")
print(f"In use:           {total - available:6.1f} GB")
print(f"Still available:  {available:6.1f} GB")


## 11. Wrap up and free the memory

You deployed a 30B-parameter Nemotron model on your Dell Pro Max GB10. It answered questions, streamed a reply, and read a long document.

Part 2 needs this system's memory, so the next cell stops the model server. The downloaded model stays on disk, so you can start the server again later without downloading it again.

**Learn more:** [Nemotron usage cookbook](https://github.com/NVIDIA-NeMo/Nemotron/tree/main/usage-cookbook)


In [ ]:
import subprocess

subprocess.run("docker rm -f vllm-nemotron-nano", shell=True, capture_output=True, text=True)
m = meminfo_gb()
print("Model server stopped.")
print(f"Memory available again: {m['MemAvailable']:.1f} GB")
